# AIC 2025 — Whisper ASR → keyframe metadata → Google Cloud Storage

Notebook này triển khai **phần ASR của pipeline Vortex**:

1. Đọc video trực tiếp từ Kaggle Dataset `aresusayhi/ai-challenge-2025`.
2. Tách audio thành **16 kHz, mono, FLAC**.
3. Chạy **Whisper** để lấy transcript có timestamp `(start, end, text)`.
4. Đọc `map-keyframes/<video_id>.csv`.
5. Với mỗi keyframe có timestamp `t_k`:
   - nếu `start <= t_k <= end` → gán transcript của segment đó;
   - nếu keyframe nằm trong khoảng im lặng → **propagate câu nói gần nhất trước đó**;
   - nếu keyframe xuất hiện trước câu nói đầu tiên → để rỗng.
6. Upload lên GCS:
   - audio FLAC;
   - ASR segment JSON;
   - keyframe-aligned ASR `.parquet` + `.jsonl`;
   - manifest `_SUCCESS` để resume an toàn.

> Lưu ý: trong Vortex, Whisper tạo **ASR metadata**, không phải vector embedding cho Milvus.  
> Vector retrieval của paper đến từ CLIP/SigLIP2. Notebook này chỉ tái hiện nhánh Whisper/ASR và alignment.

## Bảo mật credential

**Không hard-code private key vào notebook.**  
Hãy tạo một **Kaggle Private Dataset** chứa file service-account JSON của bạn, rồi attach dataset đó vào notebook.
Notebook sẽ tự tìm file có tên gần giống:

`gen-lang-client-0547522732-410672fac05f.json`

Bucket mặc định: `aic_ai_2026`.

In [ ]:
# Cài dependency. Bật Internet trong Kaggle Notebook để tải package/model lần đầu.
%pip install -q -U faster-whisper google-cloud-storage pyarrow pandas tqdm

In [ ]:
import os
import re
import json
import time
import math
import shutil
import subprocess
import traceback
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm.auto import tqdm

from google.cloud import storage
from google.oauth2 import service_account
from google.api_core.retry import Retry

from faster_whisper import WhisperModel, BatchedInferencePipeline

## 1. Cấu hình

### Model
Mặc định dùng `turbo` vì nhanh và vẫn là Whisper multilingual.  
Nếu ưu tiên accuracy tối đa hơn tốc độ, đổi thành:

```python
WHISPER_MODEL = "large-v3"
```

### Ngôn ngữ
- `LANGUAGE = None`: Whisper tự detect từng video — an toàn khi dataset có xen tiếng Anh/ngoại ngữ.
- `LANGUAGE = "vi"`: nhanh/ổn định hơn một chút nếu chắc chắn audio chủ yếu là tiếng Việt.

### Audio trên GCS
Paper chỉ cần transcript để retrieval; tuy nhiên theo yêu cầu của bạn notebook vẫn upload audio.
Nếu không muốn tốn storage/bandwidth, đặt `UPLOAD_AUDIO = False`.

In [ ]:
# ---------------------------
# DATA / GCS
# ---------------------------
KAGGLE_DATASET_ROOT = Path("/kaggle/input/ai-challenge-2025")
WORK_ROOT = Path("/kaggle/working/aic2025_asr")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

GCS_BUCKET = "aic_ai_2026"
GCS_PUBLIC_URL = "https://storage.googleapis.com/aic_ai_2026"
GCS_PREFIX = "aic2025/features/asr_whisper"

# Notebook sẽ tìm recursive JSON này trong /kaggle/input và /kaggle/working.
GCS_CREDENTIAL_HINT = "gen-lang-client-0547522732-410672fac05f"

# ---------------------------
# WHISPER
# ---------------------------
WHISPER_MODEL = "turbo"        # đổi "large-v3" nếu ưu tiên accuracy
LANGUAGE = None                # None = auto-detect; hoặc "vi"
COMPUTE_TYPE = "float16"
BATCH_SIZE = 16
BEAM_SIZE = 5
USE_VAD = True

# ---------------------------
# PIPELINE
# ---------------------------
UPLOAD_AUDIO = True
DELETE_LOCAL_AFTER_UPLOAD = True
OVERWRITE = False

MAX_VIDEOS = None              # ví dụ 3 để test; None = toàn bộ
ONLY_LEVELS = None             # ví dụ {"L21", "L22"}; None = tất cả

SCAN_DURATIONS = True
FFPROBE_WORKERS = min(16, os.cpu_count() or 4)

print("WORK_ROOT:", WORK_ROOT)

In [ ]:
def run(cmd, check=True, capture=True):
    return subprocess.run(
        cmd,
        check=check,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None,
    )

def extract_level(video_id: str):
    m = re.search(r"(L\d{2})", video_id.upper())
    return m.group(1) if m else None

def detect_dataset_root():
    if KAGGLE_DATASET_ROOT.exists():
        return KAGGLE_DATASET_ROOT

    roots = []
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for p in input_root.iterdir():
            if p.is_dir() and "challenge" in p.name.lower():
                if any(p.rglob("*.mp4")):
                    roots.append(p)
    if not roots:
        raise FileNotFoundError(
            "Không tìm thấy Kaggle dataset. Attach "
            "'aresusayhi/ai-challenge-2025' vào notebook rồi chạy lại."
        )
    return roots[0]

DATASET_ROOT = detect_dataset_root()
print("Detected DATASET_ROOT =", DATASET_ROOT)

def find_gcs_credential():
    search_roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    matches = []
    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob("*.json"):
            if GCS_CREDENTIAL_HINT.lower() in p.name.lower():
                matches.append(p)

    if not matches:
        raise FileNotFoundError(
            "Không tìm thấy service-account JSON.\n"
            "Hãy tạo Kaggle PRIVATE Dataset chứa credential, attach vào notebook, rồi chạy lại."
        )

    matches.sort(key=lambda p: ("/kaggle/input/" not in str(p), len(str(p))))
    return matches[0]

GCS_CREDENTIALS_FILE = find_gcs_credential()
print("Credential file found:", GCS_CREDENTIALS_FILE.name)

In [ ]:
# Kết nối GCS và kiểm tra bucket.
creds = service_account.Credentials.from_service_account_file(
    str(GCS_CREDENTIALS_FILE)
)
gcs_client = storage.Client(project=creds.project_id, credentials=creds)
bucket = gcs_client.bucket(GCS_BUCKET)
bucket.reload()

print("Connected project:", creds.project_id)
print("Connected bucket:", bucket.name)

GCS_RETRY = Retry(
    initial=1.0,
    maximum=30.0,
    multiplier=2.0,
    deadline=300.0,
)

def gcs_blob_name(kind: str, video_id: str, suffix: str):
    level = extract_level(video_id) or "UNKNOWN"
    return f"{GCS_PREFIX}/{kind}/{level}/{video_id}{suffix}"

def blob_exists(blob_name: str) -> bool:
    return bucket.blob(blob_name).exists(client=gcs_client)

def upload_file(local_path: Path, blob_name: str, content_type=None):
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(
        str(local_path),
        content_type=content_type,
        retry=GCS_RETRY,
        timeout=300,
    )
    return f"gs://{GCS_BUCKET}/{blob_name}"

## 2. Scan video + map-keyframes

Notebook không giả định cứng folder `Videos/videos_Lxx`; nó scan recursive `.mp4/.mkv/.webm/.mov`.

Mapping keyframe được ghép theo `video_id = Path(video).stem`, ưu tiên CSV có path chứa `map-keyframes`.

In [ ]:
VIDEO_EXTS = {".mp4", ".mkv", ".webm", ".mov", ".avi", ".m4v"}

video_paths = sorted(
    p for p in DATASET_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in VIDEO_EXTS
)

if ONLY_LEVELS:
    ONLY_LEVELS = {x.upper() for x in ONLY_LEVELS}
    video_paths = [
        p for p in video_paths
        if extract_level(p.stem) in ONLY_LEVELS
    ]

if MAX_VIDEOS is not None:
    video_paths = video_paths[:MAX_VIDEOS]

csv_paths = list(DATASET_ROOT.rglob("*.csv"))

def build_mapping_index(csvs):
    idx = {}
    for p in csvs:
        idx.setdefault(p.stem.lower(), []).append(p)
    return idx

mapping_index = build_mapping_index(csv_paths)

def find_mapping_file(video_id: str):
    candidates = mapping_index.get(video_id.lower(), [])
    if not candidates:
        return None
    candidates = sorted(
        candidates,
        key=lambda p: (
            "map-keyframe" not in str(p).lower(),
            "keyframe" not in str(p).lower(),
            len(str(p)),
        ),
    )
    return candidates[0]

print(f"Videos selected: {len(video_paths):,}")
print(f"CSV files found: {len(csv_paths):,}")
for p in video_paths[:5]:
    print(" ", p.relative_to(DATASET_ROOT), "->", find_mapping_file(p.stem))

In [ ]:
def probe_duration_sec(video_path: Path):
    try:
        r = run([
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(video_path),
        ])
        return float(r.stdout.strip())
    except Exception:
        return float("nan")

def has_audio_stream(video_path: Path):
    try:
        r = run([
            "ffprobe", "-v", "error",
            "-select_streams", "a:0",
            "-show_entries", "stream=index",
            "-of", "csv=p=0",
            str(video_path),
        ])
        return bool(r.stdout.strip())
    except Exception:
        return False

durations = {}

if SCAN_DURATIONS and video_paths:
    with ThreadPoolExecutor(max_workers=FFPROBE_WORKERS) as ex:
        futures = {ex.submit(probe_duration_sec, p): p for p in video_paths}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="ffprobe"):
            p = futures[fut]
            durations[str(p)] = fut.result()

    valid = [v for v in durations.values() if math.isfinite(v)]
    total_hours = sum(valid) / 3600
    avg_min = (sum(valid) / len(valid) / 60) if valid else float("nan")
    print(f"Total probed duration: {total_hours:.2f} hours")
    print(f"Average: {avg_min:.2f} min/video")
else:
    total_hours = float("nan")

## 3. Load Whisper

`BatchedInferencePipeline` giúp tận dụng GPU tốt hơn cho audio dài.

Nếu Kaggle báo lỗi CUDA/cuDNN ngay sau `%pip install`, hãy **Restart Session** một lần rồi chạy lại từ cell import.

In [ ]:
try:
    print(run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).stdout)
except Exception:
    print("nvidia-smi không khả dụng.")

whisper_model = WhisperModel(
    WHISPER_MODEL,
    device="cuda",
    compute_type=COMPUTE_TYPE,
)

asr_pipeline = BatchedInferencePipeline(model=whisper_model)

print("Whisper ready:", WHISPER_MODEL)

## 4. Hàm tách audio, transcribe và align transcript → keyframe

In [ ]:
def extract_audio_flac(video_path: Path, audio_out: Path):
    audio_out.parent.mkdir(parents=True, exist_ok=True)

    if not has_audio_stream(video_path):
        return False

    cmd = [
        "ffmpeg",
        "-hide_banner", "-loglevel", "error",
        "-y",
        "-i", str(video_path),
        "-vn",
        "-map", "0:a:0",
        "-ac", "1",
        "-ar", "16000",
        "-c:a", "flac",
        "-compression_level", "5",
        str(audio_out),
    ]
    run(cmd)
    return audio_out.exists() and audio_out.stat().st_size > 0


def transcribe_audio(audio_path: Path):
    segments_gen, info = asr_pipeline.transcribe(
        str(audio_path),
        batch_size=BATCH_SIZE,
        beam_size=BEAM_SIZE,
        language=LANGUAGE,
        task="transcribe",
        vad_filter=USE_VAD,
        word_timestamps=False,
    )

    segments = []
    for i, seg in enumerate(segments_gen):
        text = (seg.text or "").strip()
        segments.append({
            "id": int(i),
            "start": float(seg.start),
            "end": float(seg.end),
            "text": text,
            "avg_logprob": float(seg.avg_logprob),
            "no_speech_prob": float(seg.no_speech_prob),
        })

    return {
        "model": WHISPER_MODEL,
        "language": info.language,
        "language_probability": float(info.language_probability),
        "duration_sec": float(info.duration),
        "duration_after_vad_sec": float(getattr(info, "duration_after_vad", info.duration)),
        "full_text": " ".join(s["text"] for s in segments if s["text"]),
        "segments": segments,
    }


def load_keyframe_mapping(mapping_path: Path):
    df = pd.read_csv(mapping_path)
    original_cols = list(df.columns)
    lower = {str(c).strip().lower(): c for c in df.columns}

    time_col = None
    for c in ["pts_time", "timestamp", "time", "time_sec", "seconds", "sec"]:
        if c in lower:
            time_col = lower[c]
            break

    if time_col is not None:
        pts_time = pd.to_numeric(df[time_col], errors="coerce")
    else:
        frame_col = next(
            (lower[c] for c in ["frame_idx", "frame_index", "frame"] if c in lower),
            None,
        )
        fps_col = next(
            (lower[c] for c in ["fps", "frame_rate"] if c in lower),
            None,
        )
        if frame_col is None or fps_col is None:
            raise ValueError(
                f"Không tìm thấy pts_time hoặc cặp frame_idx/fps trong {mapping_path}. "
                f"Columns={original_cols}"
            )
        frame_idx = pd.to_numeric(df[frame_col], errors="coerce")
        fps = pd.to_numeric(df[fps_col], errors="coerce")
        pts_time = frame_idx / fps

    id_col = next(
        (
            lower[c]
            for c in ["n", "frame_id", "keyframe_id", "image_id", "filename", "file_name"]
            if c in lower
        ),
        None,
    )

    out = df.copy()
    out["pts_time"] = pts_time.astype(float)
    out["keyframe_id"] = out[id_col].astype(str) if id_col is not None else out.index.astype(str)
    return out


def align_asr_to_keyframes(mapping_df: pd.DataFrame, segments: list, video_id: str):
    # Logic Vortex:
    # - t_k in [start,end] => segment text
    # - silent gap => propagate câu nói gần nhất trước đó
    # - trước câu nói đầu tiên => empty string
    segs = sorted(segments, key=lambda x: (x["start"], x["end"]))
    work = mapping_df.copy()
    work["_orig_order"] = range(len(work))
    work = work.sort_values("pts_time", kind="stable")

    rows = []
    j = 0
    last_idx = None

    for _, r in work.iterrows():
        t = float(r["pts_time"])

        while j < len(segs) and segs[j]["end"] < t:
            last_idx = j
            j += 1

        current_idx = None
        propagated = False

        if j < len(segs) and segs[j]["start"] <= t <= segs[j]["end"]:
            current_idx = j
            last_idx = j
        elif last_idx is not None:
            current_idx = last_idx
            propagated = True

        if current_idx is None:
            text = ""
            seg_start = None
            seg_end = None
            seg_id = None
        else:
            s = segs[current_idx]
            text = s["text"]
            seg_start = s["start"]
            seg_end = s["end"]
            seg_id = s["id"]

        item = r.to_dict()
        item.update({
            "video_id": video_id,
            "asr_text": text,
            "asr_segment_id": seg_id,
            "asr_segment_start": seg_start,
            "asr_segment_end": seg_end,
            "asr_propagated": bool(propagated),
        })
        rows.append(item)

    return (
        pd.DataFrame(rows)
        .sort_values("_orig_order")
        .drop(columns=["_orig_order"])
    )

## 5. Process từng video

Output GCS mặc định:

```text
gs://aic_ai_2026/aic2025/features/asr_whisper/
  audio/Lxx/<video_id>.flac
  segments/Lxx/<video_id>.json
  keyframes/Lxx/<video_id>.parquet
  keyframes/Lxx/<video_id>.jsonl
  manifests/Lxx/<video_id>.json
```

Manifest được upload **cuối cùng**. Nếu session chết giữa chừng, lần chạy sau có thể resume và skip video đã hoàn tất.

In [ ]:
def save_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def process_video(video_path: Path):
    video_id = video_path.stem
    level = extract_level(video_id) or "UNKNOWN"

    manifest_blob = gcs_blob_name("manifests", video_id, ".json")
    if (not OVERWRITE) and blob_exists(manifest_blob):
        return {
            "video_id": video_id,
            "status": "skipped",
            "reason": "manifest_exists",
            "elapsed_sec": 0.0,
        }

    local_dir = WORK_ROOT / level / video_id
    local_dir.mkdir(parents=True, exist_ok=True)

    audio_path = local_dir / f"{video_id}.flac"
    seg_json = local_dir / f"{video_id}.segments.json"
    aligned_parquet = local_dir / f"{video_id}.keyframes.parquet"
    aligned_jsonl = local_dir / f"{video_id}.keyframes.jsonl"
    manifest_path = local_dir / f"{video_id}.manifest.json"

    t0 = time.perf_counter()
    duration_sec = durations.get(str(video_path), float("nan"))
    if not math.isfinite(duration_sec):
        duration_sec = probe_duration_sec(video_path)

    mapping_path = find_mapping_file(video_id)

    try:
        t_audio = time.perf_counter()
        has_audio = extract_audio_flac(video_path, audio_path)
        audio_elapsed = time.perf_counter() - t_audio

        if has_audio:
            t_asr = time.perf_counter()
            asr = transcribe_audio(audio_path)
            asr_elapsed = time.perf_counter() - t_asr
        else:
            asr_elapsed = 0.0
            asr = {
                "model": WHISPER_MODEL,
                "language": None,
                "language_probability": 0.0,
                "duration_sec": float(duration_sec) if math.isfinite(duration_sec) else None,
                "duration_after_vad_sec": 0.0,
                "full_text": "",
                "segments": [],
            }

        asr_doc = {
            "schema_version": 1,
            "video_id": video_id,
            "source_video": str(video_path.relative_to(DATASET_ROOT)),
            "created_at": datetime.now(timezone.utc).isoformat(),
            **asr,
        }
        save_json(seg_json, asr_doc)

        aligned_rows = 0
        if mapping_path is not None:
            mapping_df = load_keyframe_mapping(mapping_path)
            aligned_df = align_asr_to_keyframes(
                mapping_df,
                asr["segments"],
                video_id=video_id,
            )

            aligned_df.to_parquet(aligned_parquet, index=False)
            aligned_df.to_json(
                aligned_jsonl,
                orient="records",
                lines=True,
                force_ascii=False,
            )
            aligned_rows = len(aligned_df)

        uploaded = {}

        if has_audio and UPLOAD_AUDIO:
            audio_blob = gcs_blob_name("audio", video_id, ".flac")
            uploaded["audio"] = upload_file(audio_path, audio_blob, "audio/flac")

        seg_blob = gcs_blob_name("segments", video_id, ".json")
        uploaded["segments"] = upload_file(
            seg_json, seg_blob, "application/json; charset=utf-8"
        )

        if mapping_path is not None:
            pq_blob = gcs_blob_name("keyframes", video_id, ".parquet")
            jl_blob = gcs_blob_name("keyframes", video_id, ".jsonl")
            uploaded["keyframes_parquet"] = upload_file(
                aligned_parquet, pq_blob, "application/octet-stream"
            )
            uploaded["keyframes_jsonl"] = upload_file(
                aligned_jsonl, jl_blob, "application/x-ndjson"
            )

        elapsed = time.perf_counter() - t0
        effective_rtf = (
            asr_elapsed / duration_sec
            if math.isfinite(duration_sec) and duration_sec > 0
            else None
        )

        manifest = {
            "schema_version": 1,
            "status": "success",
            "video_id": video_id,
            "level": level,
            "source_video": str(video_path.relative_to(DATASET_ROOT)),
            "mapping_file": (
                str(mapping_path.relative_to(DATASET_ROOT))
                if mapping_path is not None else None
            ),
            "model": WHISPER_MODEL,
            "language_setting": LANGUAGE,
            "detected_language": asr.get("language"),
            "duration_sec": duration_sec if math.isfinite(duration_sec) else None,
            "segment_count": len(asr["segments"]),
            "aligned_keyframe_count": aligned_rows,
            "audio_uploaded": bool(has_audio and UPLOAD_AUDIO),
            "audio_extract_sec": audio_elapsed,
            "asr_inference_sec": asr_elapsed,
            "elapsed_sec": elapsed,
            "asr_rtf": effective_rtf,
            "uploaded": uploaded,
            "created_at": datetime.now(timezone.utc).isoformat(),
        }
        save_json(manifest_path, manifest)
        uploaded["manifest"] = upload_file(
            manifest_path, manifest_blob, "application/json; charset=utf-8"
        )

        if DELETE_LOCAL_AFTER_UPLOAD:
            shutil.rmtree(local_dir, ignore_errors=True)

        return {
            "video_id": video_id,
            "status": "success",
            "duration_sec": duration_sec,
            "asr_sec": asr_elapsed,
            "elapsed_sec": elapsed,
            "rtf": effective_rtf,
            "segments": len(asr["segments"]),
            "keyframes": aligned_rows,
        }

    except Exception as e:
        return {
            "video_id": video_id,
            "status": "error",
            "error": repr(e),
            "traceback": traceback.format_exc(),
            "elapsed_sec": time.perf_counter() - t0,
        }

In [ ]:
# TEST 1 VIDEO trước khi chạy full.
RUN_ONE_VIDEO_TEST = True

if RUN_ONE_VIDEO_TEST and video_paths:
    sample = video_paths[0]
    print("Testing:", sample.relative_to(DATASET_ROOT))
    test_result = process_video(sample)
    print(json.dumps(test_result, ensure_ascii=False, indent=2))

## 6. Full run + resume + ETA

- Video đã có manifest trên GCS → skip.
- Sau mỗi video, notebook cập nhật **effective RTF**.
- ETA được tính từ chính GPU/Kaggle session đang chạy, đáng tin hơn benchmark bên ngoài.

In [ ]:
RUN_FULL = False  # Sau khi test 1 video OK, đổi thành True.

run_log = []
sum_audio_sec = 0.0
sum_asr_sec = 0.0
sum_wall_success_sec = 0.0
success_count = 0
error_count = 0
skip_count = 0

if RUN_FULL:
    wall_start = time.perf_counter()

    total_selected_sec = (
        sum(v for v in durations.values() if math.isfinite(v))
        if durations else float("nan")
    )

    for i, video_path in enumerate(video_paths, start=1):
        result = process_video(video_path)
        run_log.append(result)

        status = result["status"]
        if status == "success":
            success_count += 1
            d = result.get("duration_sec")
            a = result.get("asr_sec")
            w = result.get("elapsed_sec")

            if d and math.isfinite(d):
                sum_audio_sec += d
            if a and math.isfinite(a):
                sum_asr_sec += a
            if w and math.isfinite(w):
                sum_wall_success_sec += w

        elif status == "skipped":
            skip_count += 1
        else:
            error_count += 1

        if sum_audio_sec > 0 and sum_wall_success_sec > 0:
            wall_rtf = sum_wall_success_sec / sum_audio_sec

            if math.isfinite(total_selected_sec):
                remaining_audio_sec = max(total_selected_sec - sum_audio_sec, 0)
                eta_sec = remaining_audio_sec * wall_rtf
                eta_text = f"{eta_sec/3600:.2f} h"
            else:
                eta_text = "N/A"

            print(
                f"[{i}/{len(video_paths)}] {video_path.stem}: {status} | "
                f"success={success_count}, skip={skip_count}, error={error_count} | "
                f"wall_RTF={wall_rtf:.4f} | ETA≈{eta_text}"
            )
        else:
            print(f"[{i}/{len(video_paths)}] {video_path.stem}: {status}")

    wall_elapsed = time.perf_counter() - wall_start
    print("\nDONE")
    print("Wall time (h):", wall_elapsed / 3600)
    print("Success:", success_count, "Skipped:", skip_count, "Errors:", error_count)

    run_log_path = WORK_ROOT / f"run_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    save_json(run_log_path, run_log)
    log_blob = f"{GCS_PREFIX}/run_logs/{run_log_path.name}"
    print("Run log:", upload_file(run_log_path, log_blob, "application/json; charset=utf-8"))
else:
    print("RUN_FULL=False. Hãy kiểm tra test result rồi đổi RUN_FULL=True.")

## 7. Kiểm tra output trên GCS

In [ ]:
def list_gcs_prefix(prefix, limit=30):
    blobs = gcs_client.list_blobs(GCS_BUCKET, prefix=prefix)
    for i, blob in enumerate(blobs):
        if i >= limit:
            break
        print(f"gs://{GCS_BUCKET}/{blob.name}  ({blob.size or 0:,} bytes)")

list_gcs_prefix(GCS_PREFIX, limit=30)

## 8. Chạy song song nhiều Kaggle session theo level

Đây là cách ổn định hơn việc cố chia một video lên nhiều GPU.

Notebook A:

```python
ONLY_LEVELS = {"L21", "L22", "L23", "L24", "L25"}
```

Notebook B:

```python
ONLY_LEVELS = {"L26", "L27", "L28", "L29", "L30"}
```

Vì mỗi video có manifest riêng trên GCS nên pipeline có thể resume và hạn chế duplicate work.